<a href="https://colab.research.google.com/github/pearl-yu/mist5400fall2026/blob/main/week11/lab_part1_model_to_gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MIST 5400 Week 11 Lab, Part 1: from a trained model to a web app

This notebook follows Part 1 of the handout. Cells 1 to 4 are the week 3 churn model, unchanged.
Cells 5 and 6 are the two steps from the handout: ask Gemini for a Gradio cell, then run it and share the link.

Cell 1 pulls the dataset from the course GitHub repository (the week11 folder only), so there is nothing to upload.

## The model (week 3, unchanged)

In [1]:
# Pull only the week11 folder of Pearl's GitHub repository, which has the dataset.
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/pearl-yu/mist5400fall2026.git
%cd mist5400fall2026
!git sparse-checkout set week11
%cd week11
!ls

remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 1 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 144 bytes | 144.00 KiB/s, done.
/content/mist5400fall2026
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 2 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), 81.44 KiB | 2.91 MiB/s, done.
/content/mist5400fall2026/week11
churn_data.csv			 lab_part2_agent_to_gradio.ipynb
lab_part1_model_to_gradio.ipynb


In [2]:
import pandas as pd

df = pd.read_csv("churn_data.csv")
print(df.shape)
df.head()

(7032, 19)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,Yes


In [3]:
# one hot encode every text column, drop one level each so the columns are not redundant
df_encoded = pd.get_dummies(df, drop_first=True)

X = df_encoded.drop("Churn_Yes", axis=1)
y = df_encoded["Churn_Yes"]
print(X.shape, "features after encoding")

(7032, 29) features after encoding


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

log_reg_model = LogisticRegression(max_iter=1000)
log_reg_model.fit(X_train, y_train)

print("test accuracy:", round(accuracy_score(y_test, log_reg_model.predict(X_test)), 3))

test accuracy: 0.789


In [ ]:
!pip install -q gradio

import os
import gradio as gr
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# --- Robust Setup: Ensure model and X_train are in memory ---
try:
    # Try to check if they already exist in the global scope
    _ = X_train.columns
    _ = log_reg_model
except NameError:
    print("Dependencies not found in memory. Automatically loading and training model...")

    # Check if we need to clone the repository to get the data
    data_path = "/content/mist5400fall2026/week11/churn_data.csv"
    if not os.path.exists(data_path):
        print("Cloning repository to retrieve dataset...")
        !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/pearl-yu/mist5400fall2026.git /content/mist5400fall2026
        %cd /content/mist5400fall2026
        !git sparse-checkout set week11
        %cd /content/mist5400fall2026/week11

    # Load and process dataset
    df_fallback = pd.read_csv(data_path)
    df_encoded_fallback = pd.get_dummies(df_fallback, drop_first=True)
    X_fallback = df_encoded_fallback.drop("Churn_Yes", axis=1)
    y_fallback = df_encoded_fallback["Churn_Yes"]

    X_train, X_test, y_train, y_test = train_test_split(X_fallback, y_fallback, test_size=0.2, random_state=42)
    log_reg_model = LogisticRegression(max_iter=1000)
    log_reg_model.fit(X_train, y_train)
    print("Model trained successfully!")

# Default typical values for columns not configured by the user inputs
typical_values = {
    'gender': 'Male',
    'Partner': 'No',
    'Dependents': 'No',
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'No',
    'DeviceProtection': 'No',
    'StreamingTV': 'No',
    'StreamingMovies': 'No'
}

def predict_churn(tenure, MonthlyCharges, Contract, InternetService, PaymentMethod, PaperlessBilling, SeniorCitizen, TechSupport, threshold):
    # 1. Build a single raw row dictionary with original column names
    raw_data = {
        'gender': typical_values['gender'],
        'SeniorCitizen': int(SeniorCitizen),
        'Partner': typical_values['Partner'],
        'Dependents': typical_values['Dependents'],
        'tenure': int(tenure),
        'PhoneService': typical_values['PhoneService'],
        'MultipleLines': typical_values['MultipleLines'],
        'InternetService': InternetService,
        'OnlineSecurity': typical_values['OnlineSecurity'],
        'OnlineBackup': typical_values['OnlineBackup'],
        'DeviceProtection': typical_values['DeviceProtection'],
        'TechSupport': TechSupport,
        'StreamingTV': typical_values['StreamingTV'],
        'StreamingMovies': typical_values['StreamingMovies'],
        'Contract': Contract,
        'PaperlessBilling': 'Yes' if PaperlessBilling else 'No',
        'PaymentMethod': PaymentMethod,
        'MonthlyCharges': float(MonthlyCharges)
    }

    # Convert to DataFrame
    raw_df = pd.DataFrame([raw_data])

    # 2. Apply get_dummies
    encoded_df = pd.get_dummies(raw_df, drop_first=True)

    # 3. Reindex to match the training features (X_train.columns) with fill_value=0
    encoded_df = encoded_df.reindex(columns=X_train.columns, fill_value=0)

    # 4. Predict probability
    prob = log_reg_model.predict_proba(encoded_df)[0][1]

    # 5. Make decision at the threshold
    decision = "Churn" if prob >= threshold else "No Churn"

    return f"{prob:.2%}", decision

# Gradio interface definition
demo = gr.Interface(submit_btn="Submit",
    fn=predict_churn,
    inputs=[
        gr.Slider(minimum=0, maximum=72, step=1, value=12, label="tenure (months)"),
        gr.Slider(minimum=10, maximum=150, step=0.5, value=70.0, label="MonthlyCharges"),
        gr.Dropdown(choices=["Month-to-month", "One year", "Two year"], value="Month-to-month", label="Contract"),
        gr.Dropdown(choices=["DSL", "Fiber optic", "No"], value="Fiber optic", label="InternetService"),
        gr.Dropdown(choices=["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"], value="Electronic check", label="PaymentMethod"),
        gr.Checkbox(value=True, label="PaperlessBilling"),
        gr.Checkbox(value=False, label="SeniorCitizen"),
        gr.Dropdown(choices=["No", "Yes", "No internet service"], value="No", label="TechSupport"),
        gr.Slider(minimum=0.0, maximum=1.0, step=0.05, value=0.5, label="Decision Threshold")
    ],
    outputs=[
        gr.Textbox(label="Probability of Churn"),
        gr.Textbox(label="Decision")
    ],
    live=False,
    title="Customer Churn Predictor App",
    description="Enter customer information to predict the probability that a customer will cancel their service."
)

# Defaults chosen that were not specified:
# - Used 'Male' as gender default.
# - Used 'No' for Partner and Dependents defaults.
# - Used 'Yes' for PhoneService and 'No' for remaining internet/telecom extras.
# - Set realistic min, max, and defaults for the inputs (e.g. max tenure of 72, MonthlyCharges ranges).

demo.launch(share=True, debug=True)

Dependencies not found in memory. Automatically loading and training model...
Cloning repository to retrieve dataset...
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 1 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 144 bytes | 144.00 KiB/s, done.
/content/mist5400fall2026
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 2 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), 81.44 KiB | 4.29 MiB/s, done.
/content/mist5400fall2026/week11
Model trained successfully!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cd72b860dcf6186cd9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrade

## Step 2: run it and share the link

Running the cell above prints two links. The one ending in `.gradio.live` is public: send it to a teammate's phone and move a slider.

- The link works while this notebook is open and the cell is running; Gradio switches it off after 72 hours. Rerun the cell before the final presentation.
- To change the app, edit the cell (or ask Gemini to change it) and run it again. Describe your changes in the prompt as design choices: which inputs the user can set, what the output should say, what the title and description are.
- For the permanent link due Dec 1, see "How long the link lives" in the handout.

## Submit

Submit this run-through notebook (File → Download → Download .ipynb) to eLC → Assignment → Week 11 Lab by Oct 29, end of day.

Optional: change the prompts / designs.